### Data preparation

You can build data from repository or use our data

In [ ]:
# build data, use the lab option to label data in idealized setting

!vulguard mining \
    -repo_name FFmpeg \
    -repo_path clone \
    -mode local \
    -repo_language C \
    -szz vszz \
    -dg_save_folder . \
    -workers 10

In [ ]:
# download our data from https://figshare.com/s/bedbc45f494aed760e06

!wget --content-disposition https://ndownloader.figshare.com/articles/28623650?private_link=bedbc45f494aed760e06

### Random Sampling && Advanced Sampling

In [ ]:
# Sampling

import json
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, OneSidedSelection

# -------- CONFIG --------
label_key = "label"   
sampling_strategy = "over"
path = "<DOWNLOAD_DIR>/data/FFmpeg/Realistic/"
input_file = "SETUP2-FFmpeg-features-train.jsonl"
output_file = f"{path}/{sampling_strategy}-sampling-{input_file}" 
# ------------------------

# Load JSONL into DataFrame
df = pd.read_json(input_file, lines=True, orient="records")

# Separate features & labels
X = df.drop(columns=[label_key])
y = df[label_key]

# Choose sampler
if sampling_strategy == "over":
    sampler = RandomOverSampler(random_state=42)
elif sampling_strategy == "under":
    sampler = RandomUnderSampler(random_state=42)
elif sampling_strategy == "smote":
    sampler = SMOTE(random_state=42)
elif sampling_strategy == "oss":
    sampler = OneSidedSelection(random_state=42)

# Apply sampling
X_resampled, y_resampled = sampler.fit_resample(X, y)

# Merge back into a DataFrame
resampled_df = X_resampled.copy()
resampled_df[label_key] = y_resampled

with open(output_file, "w", encoding="utf-8") as f:
    for record in resampled_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
# Training

%%bash
#Train ML_models
projects=("FFmpeg", "linux")
ml_models=("lapredict", "tlel", "lr")

for project in "${projects[@]}"; do 
    for model in "${ml_models[@]}"; do
        echo "Training: $project - $model"
        vulguard training \
            -model $model \
            -repo_name $project \
            -repo_language C \
            -dg_save_folder . \
            -train_set <DOWNLOAD_DIR>/data/$project/Idealized/over-sampling-SETUP2-$project-features-train.jsonl \
            -val_set <DOWNLOAD_DIR>/data/$project/Idealized/SETUP2-$project-features-val.jsonl \
            -dictionary <DOWNLOAD_DIR>/data/$project/Idealized/dict-$project.jsonl
    done
done

In [ ]:
# Testing

%%bash
#ML_models
projects=("FFmpeg", "linux")
ml_models=("lapredict", "tlel", "lr")

for project in "${projects[@]}"; do 
    for model in "${ml_models[@]}"; do
        echo "Testing: $project - $model"
        vulguard evaluating \
            -model $model \
            -repo_name $project \
            -repo_language C \
            -dg_save_folder . \
            -test_set <DOWNLOAD_DIR>/data/$project/Idealized/SETUP2-$project-features-test.jsonl \
            -dictionary <DOWNLOAD_DIR>/data/$project/Idealized/dict-$project.jsonl
    done
done

### Focal Loss

In [**vulguard/models/deepjit/warper.py**](../vulguard/models/deepjit/warper.py), comment line 137 and uncomment line 138.

The code should look like this:
```python
137    # loss = criterion(predict, label)
138    loss = sigmoid_focal_loss(predict, label)
```

In [ ]:
%%bash
# Train
projects=("FFmpeg", "linux")
model="deepjit"

for project in "${projects[@]}"; do 
    echo "Training: $project - $model"
    vulguard training \
        -model $model \
        -repo_name $project \
        -repo_language C \
        -dg_save_folder . \
        -device cuda \
        -train_set <DOWNLOAD_DIR>/data/$project/Realistic/SETUP2-$project-deepjit-train.jsonl \
        -val_set <DOWNLOAD_DIR>/data/$project/Realistic/SETUP2-$project-deepjit-val.jsonl \
        -dictionary <DOWNLOAD_DIR>/data/$project/Realistic/dict-$project.jsonl
done

In [ ]:
%%bash
#Test
projects=("FFmpeg", "linux")
model="deepjit"

for project in "${projects[@]}"; do 
    echo "Testing: $project - $model"
    vulguard evaluating \
        -model $model \
        -repo_name $project \
        -repo_language C \
        -dg_save_folder . \
        -device cuda \
        -test_set <DOWNLOAD_DIR>/data/$project/Realistic/SETUP2-$project-deepjit-test.jsonl \
        -dictionary <DOWNLOAD_DIR>/data/$project/Realistic/dict-$project.jsonl
done